# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import numpy as np, pandas as pd

te = d.iloc[te_i].copy()                     # grouped test split from Week 6
te["model_score"] = m.predict_proba(te[FEATS])[:, 1]

# Reason code: the signal that is most extreme for each page (percentile rank within the test split)
pcts = pd.DataFrame({f: te[f].rank(pct=True) for f in FEATS})
te["reason_code"] = pcts.idxmax(axis=1)      # rename to readable codes in the next cell

REASON_LABELS = {                            # edit these to match your real feature names
    "___feat1___": "STALE_CONTENT",
    "___feat2___": "LOW_CTR_FOR_POSITION",
    "___feat3___": "FALLING_VISIBILITY",
    "___feat4___": "___CODE___",
    "___feat5___": "___CODE___",
}
te["reason_code"] = te["reason_code"].map(REASON_LABELS)

ACTION_MAP = {                               # one action per reason code
    "STALE_CONTENT": "Refresh content",
    "LOW_CTR_FOR_POSITION": "Review title and meta",
    "FALLING_VISIBILITY": "Review and improve",
    "___CODE___": "___ACTION___",
}
te["action"] = te["reason_code"].map(ACTION_MAP).fillna("Monitor")

queue = (te.sort_values("model_score", ascending=False)
           [["___id___", "model_score", "reason_code", "action"]]
           .reset_index(drop=True))
queue["rank"] = queue.index + 1
queue.head(10)

## 1) Ranked actions and reason codes

The queue ranks pages by model score. Each row gets **one reason code** and **one action label**.

| Reason code | What it means | Action | Archetype |
|---|---|---|---|
| STALE_CONTENT | ___ | Refresh content | Improve |
| LOW_CTR_FOR_POSITION | ___ | Review title and meta | Improve |
| FALLING_VISIBILITY | ___ | Review and improve | Rewrite or improve |
| ___ | ___ | ___ | Protect / merge / prune / monitor |

**Decay and refresh insight:** ___ (one sentence from your Week 4 staleness bucket table, for example "pages not updated for the longest time carried the proxy label more often in this sample (observed, directional)").

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
STALE = "___staleness_col___"
d["stale_bucket"] = pd.qcut(d[STALE], 5, duplicates="drop")
decay = d.groupby("stale_bucket", observed=True)[LABEL].agg(n="size", rate="mean").round(3)
decay

## 2) Intended use and limits

**Intended use:** A ranked review queue for a content team. It tells reviewers where to look first. It does not decide what to change.

**Limits:**
- The label is a proxy, so a high score means "looks like pages that carried the label", not "this page needs a refresh".
- Results are measured on an anonymized sample with a grouped split, and may not hold on other sites.
- The model describes associations, not causes.
- Scores come from one month of development data, so they can drift.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3) Human review and the no-go list

**Human review rules:**
- A person reviews every page in the queue before any change is made.
- A reviewer checks the page's purpose and recent history first, and can reject any row.
- Rejected rows are logged with a reason, which becomes feedback for the next version.

**Cost and value (rough, practical):**
- Review capacity is about ___ pages per cycle, so the queue is useful only at the top.
- A false positive costs ___ minutes of reviewer time. A missed page costs ___ (a slower fix).
- At Precision@50 of ___, about ___ of every 50 reviews would hit a page carrying the proxy label, against ___ for the baseline.

**What should NOT be automated:**
- Publishing, rewriting or deleting any page based on the score alone
- Merging or pruning pages with no human check
- Reporting the score to anyone as proof of ranking impact
- Any use on sites or groups outside what I tested

## 4) Monitoring and retrain triggers

Light and non-production. Check these each cycle:

| Check | Trigger | Response |
|---|---|---|
| Precision@50 on the newest labelled month | Falls below the baseline of ___ | Pause use, investigate |
| Feature distributions (median of each feature) | Move more than ___% from development month | Review features, consider a retrain |
| Reviewer rejection rate in the top 50 | Above ___% | Revisit reason codes and the label |
| New data fields or site changes | Any | Re-run the leakage audit |

**Retrain** when a trigger fires twice in a row, using a fresh grouped split.

In [ ]:
import os, json
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Queue (stays out of git by design; regenerated on every run)
queue.to_csv("work/outputs/action_queue.csv", index=False)

# Figure 1: model vs baseline
comparison = pd.DataFrame({"Method": ["Baseline rule", "Model"],
                           "Precision@50": [___BASELINE_P50___, ___MODEL_P50___]})
ax = comparison.plot.bar(x="Method", y="Precision@50", legend=False, rot=0, figsize=(6, 4))
ax.set_ylabel("Precision@50"); ax.set_title("Model vs baseline (grouped test split)")
plt.tight_layout(); plt.savefig("work/figures/model_vs_baseline.png", dpi=150); plt.close()

# Figure 2: decay insight
decay["rate"].plot.bar(figsize=(6, 4), rot=30)
plt.ylabel("Share with proxy label"); plt.title("Label rate by staleness bucket")
plt.tight_layout(); plt.savefig("work/figures/decay_by_staleness.png", dpi=150); plt.close()

# Metrics receipt (commit this one)
json.dump({"baseline_precision_at_50": ___BASELINE_P50___,
           "model_precision_at_50": ___MODEL_P50___,
           "n_test_rows": int(len(te)),
           "split": "grouped"},
          open("work/outputs/w07_metrics.json", "w"), indent=2)
print("exported")

## 5) Exports for the paper

- `work/outputs/action_queue.csv` (regenerated each run, not committed)
- `work/outputs/w07_metrics.json` (committed receipt)
- `work/figures/model_vs_baseline.png` and `work/figures/decay_by_staleness.png` (committed)

## 6) Self-check

- [x] Intended use, ranked actions and reason codes
- [x] Human review rules and the no-go list
- [x] Cost and value thinking
- [x] Monitoring and retrain triggers
- [x] Queue, figures and metrics exported for the paper
- [x] Practical and non-production, with safe claim language
- [x] No client names, domains, URLs or tokens in any output

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.